# Exercise 4: Decision trees for the extruder blockage
**Internet of Production – AI, week 2** · about 40 min · work in pairs

**Thread:** In week 1 you built a KNN classifier that warns of an extruder blockage (exercise 2). In the first block of this week you rebuilt it in KNIME. Now you look for a model that is **easy to read and to explain to the shop floor**: a decision tree.

**Questions of this exercise**
1. Is a single threshold on the energy enough?
2. How deep may a tree be before it only memorises the training data?
3. Can the tree be read as a rule that a technician understands?
4. How does it compare with KNN?

Every `___` is a gap. The code that prepares the data is exercise 2 in one function; you do not need to change it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

DATA = Path('../../00_Data/printfarm')
HORIZON = 360                       # minutes: warn 6 hours ahead
FEATURES = ['E30', 'E120', 'slope']

def make_blockage_data(k, horizon=HORIZON):
    """The pipeline of exercise 2 of week 1 in one function: one row per 10th printing minute."""
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='%d.%m.%Y %H:%M')
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    blockages = np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].values)
    p = d[d['State'] == 'printing'].copy()
    p['E'] = p['Energy'] * 1000                              # Wh per minute
    idx = p.index.values
    nxt = np.searchsorted(blockages, idx)
    mtb = np.where(nxt < len(blockages), (blockages[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
    p['y'] = ((mtb > 0) & (mtb <= horizon)).astype(int)
    p['E30'] = p['E'].rolling(30).mean()
    p['E120'] = p['E'].rolling(120).mean()
    p['slope'] = p['E120'] - p['E120'].shift(240)
    p['Temp_Extruder_m'] = p['Temp_Extruder'].rolling(120).mean()
    p['Temp_Bed_m'] = p['Temp_Bed'].rolling(120).mean()
    return p.dropna(subset=FEATURES + ['Temp_Extruder_m']).iloc[::10], pd.to_datetime(blockages)

def report(name, y_true, y_pred, score=None):
    auc = f'   AUC {roc_auc_score(y_true, score):5.3f}' if score is not None else ''
    print(f'{name:36s} accuracy {accuracy_score(y_true, y_pred):6.1%}   recall {recall_score(y_true, y_pred, zero_division=0):6.1%}   precision {precision_score(y_true, y_pred, zero_division=0):6.1%}{auc}')

data, blockages = make_blockage_data(1)
cut = int(len(data) * 0.7)                      # split by time: first 70 % train, last 30 % test
train, test = data.iloc[:cut], data.iloc[cut:]
X_train, y_train, X_test, y_test = train[FEATURES], train['y'], test[FEATURES], test['y']
print(len(data), 'samples,', f"{data['y'].mean():.1%} class 1; train {len(train)}, test {len(test)}")

## Step 1: A rule by hand
Plot the distribution of `E120` (mean energy of the last 2 hours) for both classes in the **training data**. Where would you put a threshold?

In [ ]:
lo, hi = np.percentile(train['E120'], [0.5, 99.5])
bins = np.linspace(lo, hi, 60)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(train.loc[train['y'] == 0, 'E120'], bins=bins, density=True, alpha=.6, color='#009EE0', label='no blockage within 6 h')
ax.hist(train.loc[train['y'] == 1, 'E120'], bins=bins, density=True, alpha=.6, color='#EE7F00', label='blockage within 6 h')
ax.set_xlabel('E120 (Wh/min)'); ax.set_ylabel('density'); ax.legend(); plt.show()

In [ ]:
THRESH = ___                                    # your choice from the plot (Wh/min)
rule = (test['E120'] > THRESH).astype(int)
report(f'rule: E120 > {THRESH}', y_test, rule, test['E120'])

## Step 2: Let the tree find the threshold
A tree of depth 1 (a *stump*) searches the best single threshold itself. Does it find yours?

In [ ]:
stump = DecisionTreeClassifier(max_depth=___, random_state=0).fit(X_train, y_train)
print(export_text(stump, feature_names=FEATURES, decimals=3))
report('decision stump', y_test, stump.predict(X_test), stump.predict_proba(X_test)[:, 1])

## Step 3: How deep may the tree be?
Train trees of increasing depth (`None` = grow until every leaf is pure). Compare the result on the **training** data with the result on the **test** data.

In [ ]:
depths = [1, 2, 3, 4, 6, 8, 12, ___]
rows = []
for depth in depths:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    rows.append({'depth': str(depth), 'leaves': t.get_n_leaves(),
                 'AUC train': roc_auc_score(y_train, t.predict_proba(X_train)[:, 1]),
                 'AUC test': roc_auc_score(y_test, t.predict_proba(X_test)[:, 1]),
                 'recall test': recall_score(y_test, t.predict(X_test)),
                 'precision test': precision_score(y_test, t.predict(X_test), zero_division=0)})
res = pd.DataFrame(rows).set_index('depth')
display(res.round(3))
res[['AUC train', 'AUC test']].plot(marker='o', figsize=(7, 4), ylabel='AUC'); plt.show()

**Describe the curves. Which depth would you choose, and why not the deepest tree?**

*Your answer:*

## Step 4: A tree you can read
Limit the tree so that every leaf has at least 200 samples (`min_samples_leaf`) and give the rare class more weight (`class_weight='balanced'`). Then print the rules.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=___, class_weight=___, random_state=0).fit(X_train, y_train)
print(export_text(tree, feature_names=FEATURES, decimals=3))
fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(tree, feature_names=FEATURES, class_names=['ok', 'warn'], filled=True, rounded=True, fontsize=8, ax=ax); plt.show()
report('tree, depth 3, balanced', y_test, tree.predict(X_test), tree.predict_proba(X_test)[:, 1])

**Write the rule of the tree in one sentence for a technician. Which feature carries the decision?**

*Your answer:*

In [ ]:
pd.Series(tree.feature_importances_, index=FEATURES).sort_values().plot.barh(figsize=(6, 2.5), color='#003A79', title='feature importance')
plt.show()

## Step 5: The threshold is a business decision
Move the threshold on `E120` and watch recall and precision. A **low** threshold warns early and often; a **high** threshold warns rarely but reliably.

In [ ]:
rows = []
for th in [1.66, 1.67, 1.68, 1.69, ___]:
    pred = (test['E120'] > th).astype(int)
    rows.append({'threshold': th, 'recall': recall_score(y_test, pred), 'precision': precision_score(y_test, pred, zero_division=0),
                 'warned samples': int(pred.sum()), 'false alarm samples': int(((pred == 1) & (y_test == 0)).sum())})
display(pd.DataFrame(rows).set_index('threshold').round(3))

**Which threshold do you recommend if a check of the nozzle takes 10 minutes and a blockage costs a failed job and about 12 hours of downtime? Which one if the technician is on call only once per shift?**

*Your answer:*

## Step 6: Comparison with KNN
All models on the same test period.

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)).fit(X_train, y_train)
report('KNN, k = 25', y_test, knn.predict(X_test), knn.predict_proba(X_test)[:, 1])
report(f'rule: E120 > {THRESH}', y_test, rule, test['E120'])
report('decision stump', y_test, stump.predict(X_test), stump.predict_proba(X_test)[:, 1])
report('tree, depth 3, balanced', y_test, tree.predict(X_test), tree.predict_proba(X_test)[:, 1])

**Which model would you hand to the shop floor, and why? What do all models have in common?**

*Your answer:*

## Step 7: Export for KNIME (already done for you)
The file used in the KNIME workflow was created like this. You can re-create it for printer 2 (for the transfer test in week 3).

In [ ]:
def export_for_knime(k, path=None):
    d, _ = make_blockage_data(k)
    d = d.assign(label=np.where(d['y'] == 1, 'warn', 'ok'), split=np.where(np.arange(len(d)) < int(len(d) * 0.7), 'train', 'test'))
    out = d[FEATURES + ['y', 'label', 'split']].round(6).reset_index()
    out.to_csv(path or f'blockage_features_printer{k}.csv', index=False)      # comma separator, decimal point (KNIME default)
    return out

export_for_knime(1).head()